# DP2 Alert (DIASource) Positions on the LSSTCam Focal Plane (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-28
**Last Modified:** 2026-07-28
**Status:** In Progress
**Keywords:** DP2, alerts, DIASource, focal plane, detector defects, dust, difference imaging

## Description

Map DP2 alert (DIASource) detections in **detector pixel coordinates** and in
**focal-plane coordinates** to test whether static pixel-/optics-level artifacts
(hot pixels, bad columns, detector-surface dust, out-of-focus optics/window dust
donuts) are generating spurious alerts.

Key functionality:
1. Query the DP2 `DiaSource` table via the RSP TAP service (row-limited for testing,
   or full-table `GROUP BY` aggregation in qserv for production).
2. Stack detections into per-detector 2D pixel histograms. Under dithering, real
   transients wash out while detector-fixed artifacts pile up at fixed (x, y).
3. Assemble a full focal-plane mosaic using `lsst.afw.cameraGeom`, optionally split
   by band to separate detector-fixed features from band-dependent optics/window dust.

**Output:** per-detector pixel-histogram grid and a focal-plane mosaic PNG in `output/`.

**Runs on:** the CLOUD RSP at data.lsst.cloud (early DP2, released 2026-07-27).
The DP2 catalogs are not on the USDF RSP as of the early release.

**Based on:** dp2.lsst.io DIASource catalog docs; sdm-schemas.lsst.io DP2 schema.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-28 | Aaron Roodman | Initial version — row-limited test scaffold + focal-plane mosaic |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters -- All configurable values collected here
# ============================================================

# --- TAP query ---
tap_schema  = "dp2"          # confirm via the discovery cell below.
                             # Early DP2 (2026-07-27) is on the CLOUD RSP
                             # (data.lsst.cloud) -- NOT USDF. A "not found in
                             # TapSchema" error means you are on the wrong RSP.
tap_table   = "DiaSource"    # ~1.0e9 rows, ~90 cols

# TESTING SWITCH: start with a row limit to develop the code cheaply.
row_limit   = 100_000        # cap rows pulled (client-side binning). Set None for no cap.
use_server_aggregation = False  # False -> pull row_limit rows and bin here (TEST)
                                # True  -> FLOOR() GROUP BY histogram in qserv (FULL run;
                                #          ignores row_limit, runs as an async job)

# --- filters applied in the WHERE clause ---
reliability_min = None       # e.g. 0.9 keeps likely-real only; None = no cut
bands           = None       # e.g. ["r"] or ["g","r","i"]; None = all bands
exclude_corner_rafts = True  # drop wavefront/guider CCDs (detector_id >= 189)

# --- binning / geometry ---
pixel_bin  = 64              # detector pixels per histogram bin
npix       = 4096            # nominal detector size (overridden per-CCD from cameraGeom)
split_by_band = False        # if True, keep per-band histograms (optics/window dust test)

# --- pixel-histogram grid to eyeball (a single raft = 9 science CCDs) ---
detectors_to_show = list(range(0, 9))

from pathlib import Path
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
tag = f"binned{pixel_bin}"

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# --- RSP / LSST stack (guarded so the notebook still imports off-RSP) ---
ON_RSP = True
try:
    from lsst.rsp import get_tap_service
    from lsst.obs.lsst import LsstCam
    from lsst.afw.cameraGeom import FOCAL_PLANE, PIXELS
except Exception as exc:  # pragma: no cover - local dev
    ON_RSP = False
    print(f"LSST stack not available ({exc!r}); run this on the RSP.")

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

<a id='functions'></a>
## Helper Functions

In [ ]:
def build_where(reliability_min=None, bands=None, exclude_corner_rafts=True):
    """Assemble the shared ADQL WHERE clause from the filter parameters."""
    clauses = []
    if exclude_corner_rafts:
        clauses.append("detector < 189")
    if reliability_min is not None:
        clauses.append(f"reliability >= {reliability_min}")
    if bands:
        band_list = ", ".join(f"'{b}'" for b in bands)
        clauses.append(f"band IN ({band_list})")
    return ("WHERE " + " AND ".join(clauses)) if clauses else ""


def build_row_query(schema, table, row_limit=None, **filt):
    """Pull individual DIASources (pixel centroid + metadata). For testing."""
    top = f"TOP {int(row_limit)} " if row_limit else ""
    return (
        f"SELECT {top}diaSourceId, detector, band, x, y, ra, dec, "
        f"reliability, midpointMjdTai\n"
        f"FROM {schema}.{table}\n"
        f"{build_where(**filt)}"
    )


def build_agg_query(schema, table, pixel_bin, **filt):
    """Server-side 2D pixel histogram via FLOOR() GROUP BY. For the full run.

    Returns one row per (detector, band, xbin, ybin) with the count and the
    mean reliability in that bin -- ~10^5-10^6 rows instead of ~10^9.
    """
    b = int(pixel_bin)
    return (
        f"SELECT detector, band, "
        f"FLOOR(x/{b}) AS xbin, FLOOR(y/{b}) AS ybin, "
        f"COUNT(*) AS n, AVG(reliability) AS rel_mean\n"
        f"FROM {schema}.{table}\n"
        f"{build_where(**filt)}\n"
        f"GROUP BY detector, band, FLOOR(x/{b}), FLOOR(y/{b})"
    )


def run_query(service, query, async_=False):
    """Run an ADQL query. Use async_=True for the full-table aggregation
    (the sync endpoint will time out on a ~10^9-row scan)."""
    print(query)
    if async_:
        job = service.submit_job(query)
        job.run()
        job.wait(phases=["COMPLETED", "ERROR", "ABORTED"])
        print("TAP job phase:", job.phase)
        if job.phase != "COMPLETED":
            raise RuntimeError(f"TAP job did not complete: {job.phase}")
        result = job.fetch_result()
    else:
        result = service.search(query)
    return result.to_table().to_pandas()

In [ ]:
def add_bins(df, pixel_bin):
    """Client-side binning for the row-mode (test) path: add xbin/ybin/n."""
    out = df.copy()
    out["xbin"] = np.floor(out["x"] / pixel_bin).astype(int)
    out["ybin"] = np.floor(out["y"] / pixel_bin).astype(int)
    out["n"] = 1
    return out


def counts_by_detector(df, pixel_bin, npix, camera=None, split_by_band=False):
    """Collapse a (detector, xbin, ybin, n) frame into per-detector 2D arrays.

    Returns dict: detector_id -> 2D count array (indexed [xbin, ybin]).
    If split_by_band, returns dict: (detector_id, band) -> 2D array.
    """
    out = {}
    key_cols = ["detector", "band"] if split_by_band else ["detector"]
    for key, g in df.groupby(key_cols):
        # pandas returns a tuple key when grouping on a list of columns.
        key = key if isinstance(key, tuple) else (key,)
        det_id = int(key[0])
        nx = ny = int(np.ceil(npix / pixel_bin))
        if camera is not None:  # size the grid to the real CCD bbox
            bbox = camera[det_id].getBBox()
            nx = int(np.ceil(bbox.getWidth() / pixel_bin))
            ny = int(np.ceil(bbox.getHeight() / pixel_bin))
        arr = np.zeros((nx, ny), dtype=float)
        xb = g["xbin"].to_numpy()
        yb = g["ybin"].to_numpy()
        n = g["n"].to_numpy()
        m = (xb >= 0) & (xb < nx) & (yb >= 0) & (yb < ny)
        np.add.at(arr, (xb[m], yb[m]), n[m])
        out_key = (det_id, key[1]) if split_by_band else det_id
        out[out_key] = arr
    return out

In [ ]:
def plot_detector_grid(counts, detectors, ncols=3, norm=None, title=""):
    """Grid of per-detector pixel histograms -- eyeball a raft for defects."""
    dets = [d for d in detectors if d in counts]
    if not dets:
        print("no detectors to show"); return None
    nrows = int(np.ceil(len(dets) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows),
                             squeeze=False)
    vmax = max(c.max() for c in counts.values()) or 1
    norm = norm or LogNorm(vmin=1, vmax=vmax)
    for ax, det in zip(axes.flat, dets):
        ax.imshow(counts[det].T, origin="lower", norm=norm, aspect="equal",
                  cmap="viridis")
        ax.set_title(f"det {det}")
        ax.set_xlabel("x bin"); ax.set_ylabel("y bin")
    for ax in axes.flat[len(dets):]:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def plot_focal_plane(counts, camera, norm=None, title="", cmap="viridis"):
    """Assemble per-detector histograms into a focal-plane mosaic (mm).

    NOTE: each CCD image is placed at its focal-plane bounding box. This is
    exact for the axis-aligned science array; sensor internal orientation
    (90-deg raft rotations) is not corrected here -- fine for spotting fixed
    hotspots, revisit if precise pixel<->FP registration is needed.
    """
    fig, ax = plt.subplots(figsize=(11, 11))
    vmax = max(c.max() for c in counts.values()) or 1
    norm = norm or LogNorm(vmin=1, vmax=vmax)
    im = None
    for det_id, arr in counts.items():
        det = camera[int(det_id)]
        corners = det.getCorners(FOCAL_PLANE)  # mm
        xs = [c.getX() for c in corners]
        ys = [c.getY() for c in corners]
        extent = [min(xs), max(xs), min(ys), max(ys)]
        im = ax.imshow(arr.T, origin="lower", extent=extent, norm=norm,
                       cmap=cmap, aspect="equal")
    ax.set_xlim(-350, 350); ax.set_ylim(-350, 350)
    ax.set_xlabel("focal plane x (mm)"); ax.set_ylabel("focal plane y (mm)")
    ax.set_title(title)
    if im is not None:
        fig.colorbar(im, ax=ax, shrink=0.8, label="DIASource count / bin")
    return fig

<a id='data'></a>
## Data Access

In [ ]:
# --- Discover the real schema/table names (do not trust the default). ---
# Early DP2 is served on the CLOUD RSP (data.lsst.cloud), not USDF, as of the
# 2026-07-27 early release. "Table [dp2.DiaSource] is not found in TapSchema"
# almost always means you are on the wrong RSP.
service = get_tap_service("tap")

schemas = service.search(
    "SELECT schema_name FROM tap_schema.schemas ORDER BY schema_name"
).to_table()
print("Schemas visible on this TAP service:")
print(schemas)

hits = service.search(
    "SELECT schema_name, table_name FROM tap_schema.tables "
    "WHERE table_name LIKE '%DiaSource%'"
).to_table()
print("\nTables matching DiaSource:")
print(hits)

if len(hits):
    tap_schema = str(hits["schema_name"][0])
    tap_table = str(hits["table_name"][0]).split(".")[-1]
    print(f"\n-> using {tap_schema}.{tap_table}")
else:
    print("\nNo DiaSource table here -- switch to the cloud RSP (data.lsst.cloud).")

In [ ]:
camera = LsstCam.getCamera()

filt = dict(reliability_min=reliability_min, bands=bands,
            exclude_corner_rafts=exclude_corner_rafts)

if use_server_aggregation:
    query = build_agg_query(tap_schema, tap_table, pixel_bin, **filt)
    df = run_query(service, query, async_=True)      # full-table -> async
else:
    query = build_row_query(tap_schema, tap_table, row_limit=row_limit, **filt)
    df = run_query(service, query, async_=False)      # row-limited test
    df = add_bins(df, pixel_bin)

print(f"rows returned: {len(df):,}")
df.head()

<a id='analysis'></a>
## Analysis

In [ ]:
# Collapse to per-detector 2D count arrays (summed over bands unless split).
counts = counts_by_detector(df, pixel_bin, npix, camera=camera,
                            split_by_band=split_by_band)

n_det = len({k[0] if isinstance(k, tuple) else k for k in counts})
total = int(sum(c.sum() for c in counts.values()))
print(f"{n_det} detectors populated, {total:,} detections binned")

# Quick look at the most-populated bins -- fixed-position spikes are the
# artifact candidates (real transients dither away).
peaks = []
for key, arr in counts.items():
    det = key[0] if isinstance(key, tuple) else key
    ix, iy = np.unravel_index(np.argmax(arr), arr.shape)
    peaks.append((det, int(ix), int(iy), float(arr[ix, iy])))
peaks = pd.DataFrame(peaks, columns=["detector", "xbin", "ybin", "peak_count"])
peaks.sort_values("peak_count", ascending=False).head(15)

<a id='results'></a>
## Results & Plots

In [ ]:
# For split_by_band=True, sum bands for these summary plots.
if split_by_band:
    fp_counts = {}
    for (det, band), arr in counts.items():
        fp_counts[det] = fp_counts.get(det, np.zeros_like(arr)) + arr
else:
    fp_counts = counts

fig1 = plot_detector_grid(fp_counts, detectors_to_show,
                          title=f"DIASource pixel histograms ({tag})")
if fig1:
    fig1.savefig(output_dir / f"alerts_detector_grid_{tag}.png",
                 dpi=120, bbox_inches="tight")

fig2 = plot_focal_plane(fp_counts, camera,
                        title=f"DP2 DIASource focal-plane stack ({tag})")
fig2.savefig(output_dir / f"alerts_focal_plane_{tag}.png",
             dpi=120, bbox_inches="tight")
plt.show()

### Next steps

- Flip `use_server_aggregation = True` (async) for the full ~10^9-row stack.
- Set `split_by_band = True` and compare per-band focal-plane mosaics: detector-fixed
  defects are band-independent; optics/window **dust donuts** are band-dependent and
  fixed in focal-plane (not pixel) coordinates.
- Overlay the LSSTCam `flat` dust map / `defects` mask (Butler) on the hotspots to
  confirm a pileup is a known artifact rather than a crowded field.
- Split by `reliability` threshold: does the real/bogus classifier already suppress
  the fixed-position pileups, or are dust spots leaking into "real" alerts?